# Fine-Tune a Generative AI Model for Dialogue Summarization (35 Points)

In this notebook, you will fine-tune an existing LLM from Hugging Face for enhanced dialogue summarization. You will use the [FLAN-T5](https://huggingface.co/docs/transformers/model_doc/flan-t5) model, which provides a high quality instruction tuned model and can summarize text out of the box. To improve the inferences, you will explore a full fine-tuning approach and evaluate the results with ROUGE metrics. Then you will perform Parameter Efficient Fine-Tuning (PEFT), evaluate the resulting model and see that the benefits of PEFT outweigh the slightly-lower performance metrics.

# Table of Contents

- [ 1 - Load Required Dependencies, Dataset and LLM](#1)
  - [ 1.1 - Set up Required Dependencies](#1.1)
  - [ 1.2 - Load Dataset and LLM](#1.2)
  - [ 1.3 - Test the Model with Zero Shot Inferencing](#1.3)
- [ 2 - Perform Full Fine-Tuning](#2)
  - [ 2.1 - Preprocess the Dialog-Summary Dataset](#2.1)
  - [ 2.2 - Fine-Tune the Model with the Preprocessed Dataset](#2.2)
  - [ 2.3 - Evaluate the Model Qualitatively (Human Evaluation)](#2.3)
  - [ 2.4 - Evaluate the Model Quantitatively (with ROUGE Metric)](#2.4)
- [ 3 - Perform Parameter Efficient Fine-Tuning (PEFT)](#3)
  - [ 3.1 - Setup the PEFT/LoRA model for Fine-Tuning](#3.1)
  - [ 3.2 - Train PEFT Adapter](#3.2)
  - [ 3.3 - Evaluate the Model Qualitatively (Human Evaluation)](#3.3)
  - [ 3.4 - Evaluate the Model Quantitatively (with ROUGE Metric)](#3.4)

<a name='1'></a>
## 1 - Load Required Dependencies, Dataset and LLM (10 points)

<a name='1.1'></a>
### 1.1 - Set up Required Dependencies (2 point)

Now install the required packages for the LLM and datasets.



In [ ]:
!pip install -q datasets torch transformers evaluate rouge_score loralib peft


Import the necessary components. Some of them are new for this week, they will be discussed later in the notebook.

In [ ]:
import os
import time

import evaluate
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

WORKDIR = "."

FULL_FT_CHECKPOINT_CANDIDATES = [
    os.path.join(WORKDIR, checkpoint_name)
    for checkpoint_name in ["flan-diaglogue-summary-checkpoint", "flan-dialogue-summary-checkpoint"]
]

PEFT_CHECKPOINT_CANDIDATES = [
    os.path.join(WORKDIR, checkpoint_name)
    for checkpoint_name in ["peft-dialogue-summary-checkpoint-from-s3", "peft-dialogue-summary-checkpoint-local"]
]

RESULTS_CSV = os.path.join(WORKDIR, "dialogue-summary-training-results.csv")


def pick_path(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]


def load_seq2seq_model(model_path):
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path, torch_dtype=model_dtype)
    return model.to(device)


def build_prompt(dialogue):
    return f"""Summarize the following conversation.

{dialogue}

Summary: """


def generate_summary(model, prompt, max_new_tokens=200):
    input_ids = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).input_ids.to(device)
    outputs = model.generate(input_ids=input_ids, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


<a name='1.2'></a>
### 1.2 - Load Dataset and LLM (4 points)

You are going to continue experimenting with the [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) Hugging Face dataset. It contains 10,000+ dialogues with the corresponding manually labeled summaries and topics.

In [ ]:
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

dataset


Load the pre-trained [FLAN-T5 model](https://huggingface.co/docs/transformers/model_doc/flan-t5) and its tokenizer directly from HuggingFace. Notice that you will be using the [small version](https://huggingface.co/google/flan-t5-small) of FLAN-T5. Setting `torch_dtype=torch.bfloat16` specifies the memory type to be used by this model.

In [ ]:
model_name = "google/flan-t5-small"

original_model = load_seq2seq_model(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


It is possible to pull out the number of model parameters and find out how many of them are trainable. The following function can be used to do that, at this stage, you do not need to go into details of it.

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0

    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()

    return (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: "
        f"{100 * trainable_model_params / all_model_params:.2f}%"
    )


print(print_number_of_trainable_model_parameters(original_model))


<a name='1.3'></a>
### 1.3 - Test the Model with Zero Shot Inferencing (2 Points)

Test the model with the zero shot inferencing. You can see that the model struggles to summarize the dialogue compared to the baseline summary, but it does pull out some important information from the text which indicates the model can be fine-tuned to the task at hand.

In [ ]:
index = 200

dialogue = dataset["test"][index]["dialogue"]
summary = dataset["test"][index]["summary"]

prompt = build_prompt(dialogue)
output = generate_summary(original_model, prompt)

dash_line = "-".join("" for x in range(100))
print(dash_line)
print(f"INPUT PROMPT:\n{prompt}")
print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}\n")
print(dash_line)
print(f"MODEL GENERATION - ZERO SHOT:\n{output}")


<a name='2'></a>
## 2 - Perform Full Fine-Tuning (15 points)

<a name='2.1'></a>
### 2.1 - Preprocess the Dialog-Summary Dataset (3 points)

You need to convert the dialog-summary (prompt-response) pairs into explicit instructions for the LLM. Prepend an instruction to the start of the dialog with `Summarize the following conversation` and to the start of the summary with `Summary` as follows:

Training prompt (dialogue):
```
Summarize the following conversation.

    Chris: This is his part of the conversation.
    Antje: This is her part of the conversation.
    
Summary:
```

Training response (summary):
```
Both Chris and Antje participated in the conversation.
```

Then preprocess the prompt-response dataset into tokens and pull out their `input_ids` (1 per token).

In [ ]:
def tokenize_function(example):
    prompts = [build_prompt(dialogue) for dialogue in example["dialogue"]]
    model_inputs = tokenizer(
        prompts,
        max_length=512,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=example["summary"],
        max_length=128,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels["input_ids"]
    ]
    return model_inputs


# The dataset actually contains 3 diff splits: train, validation, test.
# The tokenize_function code is handling all data across all splits in batches.
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(dataset["train"].column_names)
tokenized_datasets.set_format("torch")


To save some time in the lab, you will subsample the dataset:

In [ ]:
tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 100 == 0, with_indices=True)

Check the shapes of all three parts of the dataset:

In [ ]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

print(tokenized_datasets)

The output dataset is ready for fine-tuning.

<a name='2.2'></a>
### 2.2 - Fine-Tune the Model with the Preprocessed Dataset (4 points)

Now utilize the built-in Hugging Face `Trainer` class (see the documentation [here](https://huggingface.co/docs/transformers/main_classes/trainer)). Pass the preprocessed dataset with reference to the original model. Other training parameters are found experimentally and there is no need to go into details about those at the moment.

In [ ]:
output_dir = f"./dialogue-summary-training-{str(int(time.time()))}"

training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=1,
    max_steps=1,
    report_to="none",
)

trainer = Trainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)


Start training process...



The code trainer.train() utilizes the Weights & Biases (wandb) library to track and visualize the training process. To proceed, you'll need to sign up for a wandb account using your Gmail and then enter your unique API token to authenticate and enable logging of the training progress.

In [ ]:
trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Saved the 1-step fine-tuning checkpoint to: {output_dir}")
print("Later evaluation cells will prefer the provided Drive checkpoint when it is available.")


Create an instance of the `AutoModelForSeq2SeqLM` class for the instruct model:

In [ ]:
# Keep any provided checkpoints in subdirectories of the current working directory.
# Expected local names are shown below so later cells can load them without extra path edits.
print("Current working directory:", os.getcwd())
print("Full fine-tuned checkpoint candidates:", FULL_FT_CHECKPOINT_CANDIDATES)
print("PEFT checkpoint candidates:", PEFT_CHECKPOINT_CANDIDATES)
print("Results CSV:", RESULTS_CSV)


In [ ]:
# Load tokenizer and models
model_path = pick_path(FULL_FT_CHECKPOINT_CANDIDATES + [output_dir])

tokenizer = AutoTokenizer.from_pretrained(model_name)
original_model = load_seq2seq_model(model_name)
instruct_model = load_seq2seq_model(model_path)

print(f"Using full fine-tuned checkpoint: {model_path}")


<a name='2.3'></a>
### 2.3 - Evaluate the Model Qualitatively (Human Evaluation) (4 points)

As with many GenAI applications, a qualitative approach where you ask yourself the question "Is my model behaving the way it is supposed to?" is usually a good starting point. In the example below (the same one we started this notebook with), you can see how the fine-tuned model is able to create a reasonable summary of the dialogue compared to the original inability to understand what is being asked of the model.

In [ ]:
index = 200
dialogue = dataset["test"][index]["dialogue"]
human_baseline_summary = dataset["test"][index]["summary"]

prompt = build_prompt(dialogue)

original_model_text_output = generate_summary(original_model, prompt)
instruct_model_text_output = generate_summary(instruct_model, prompt)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{human_baseline_summary}")
print(dash_line)
print(f"ORIGINAL MODEL:\n{original_model_text_output}")
print(dash_line)
print(f"INSTRUCT MODEL:\n{instruct_model_text_output}")


<a name='2.4'></a>
### 2.4 - Evaluate the Model Quantitatively (with ROUGE Metric) (4 points)

The [ROUGE metric](https://en.wikipedia.org/wiki/ROUGE_(metric)) helps quantify the validity of summarizations produced by models. It compares summarizations to a "baseline" summary which is usually created by a human. While not perfect, it does indicate the overall increase in summarization effectiveness that we have accomplished by fine-tuning.

In [ ]:
rouge = evaluate.load("rouge")


Generate the outputs for the sample of the test dataset (only 10 dialogues and summaries to save time), and save the results.

In [ ]:
dialogues = dataset["test"][0:10]["dialogue"]
human_baseline_summaries = dataset["test"][0:10]["summary"]

original_model_summaries = []
instruct_model_summaries = []

for dialogue in dialogues:
    prompt = build_prompt(dialogue)
    original_model_summaries.append(generate_summary(original_model, prompt))
    instruct_model_summaries.append(generate_summary(instruct_model, prompt))

zipped_summaries = list(
    zip(human_baseline_summaries, original_model_summaries, instruct_model_summaries)
)

df = pd.DataFrame(
    zipped_summaries,
    columns=["human_baseline_summaries", "original_model_summaries", "instruct_model_summaries"],
)
df


Evaluate the models computing ROUGE metrics. Notice the improvement in the results!

In [ ]:
original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)


The file `dialogue-summary-training-results.csv` contains a pre-populated list of all model results which you can use to evaluate on a larger section of data. The next cell reads that file directly from the current working directory.


In [ ]:
results_path = RESULTS_CSV
results = pd.read_csv(results_path).drop(columns=["Unnamed: 0"], errors="ignore")

human_baseline_summaries = results["human_baseline_summaries"].values
original_model_summaries = results["original_model_summaries"].values
instruct_model_summaries = results["instruct_model_summaries"].values

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

print(f"Loaded results from: {results_path}")
print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)


The results show substantial improvement in all ROUGE metrics:

In [ ]:
print("Absolute percentage improvement of INSTRUCT MODEL over ORIGINAL MODEL")

improvement = (np.array(list(instruct_model_results.values())) - np.array(list(original_model_results.values())))
for key, value in zip(instruct_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

<a name='3'></a>
## 3 - Perform Parameter Efficient Fine-Tuning (PEFT) (10 points)

Now, let's perform **Parameter Efficient Fine-Tuning (PEFT)** fine-tuning as opposed to "full fine-tuning" as you did above. PEFT is a form of instruction fine-tuning that is much more efficient than full fine-tuning - with comparable evaluation results as you will see soon.

PEFT is a generic term that includes **Low-Rank Adaptation (LoRA)** and prompt tuning (which is NOT THE SAME as prompt engineering!). In most cases, when someone says PEFT, they typically mean LoRA. LoRA, at a very high level, allows the user to fine-tune their model using fewer compute resources (in some cases, a single GPU). After fine-tuning for a specific task, use case, or tenant with LoRA, the result is that the original LLM remains unchanged and a newly-trained “LoRA adapter” emerges. This LoRA adapter is much, much smaller than the original LLM - on the order of a single-digit % of the original LLM size (MBs vs GBs).  

That said, at inference time, the LoRA adapter needs to be reunited and combined with its original LLM to serve the inference request.  The benefit, however, is that many LoRA adapters can re-use the original LLM which reduces overall memory requirements when serving multiple tasks and use cases.

<a name='3.1'></a>
### 3.1 - Setup the PEFT/LoRA model for Fine-Tuning (2 points)

You need to set up the PEFT/LoRA model for fine-tuning with a new layer/parameter adapter. Using PEFT/LoRA, you are freezing the underlying LLM and only training the adapter. Have a look at the LoRA configuration below. Note the rank (`r`) hyper-parameter, which defines the rank/dimension of the adapter to be trained.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,  # FLAN-T5
)


Add LoRA adapter layers/parameters to the original LLM to be trained.

In [ ]:
peft_model = get_peft_model(load_seq2seq_model(model_name), lora_config)
print(print_number_of_trainable_model_parameters(peft_model))


<a name='3.2'></a>
### 3.2 - Train PEFT Adapter (3 points)

Define training arguments and create `Trainer` instance.

In [ ]:
output_dir = f"./peft-dialogue-summary-training-{str(int(time.time()))}"

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    num_train_epochs=1,
    logging_steps=1,
    max_steps=1,
    report_to="none",
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)


Now everything is ready to train the PEFT adapter and save the model.



In [ ]:
peft_trainer.train()

peft_model_path = "./peft-dialogue-summary-checkpoint-local"

peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

print(f"Saved the 1-step PEFT adapter checkpoint to: {peft_model_path}")
print("Later evaluation cells will prefer the provided Drive PEFT checkpoint when it is available.")


That training was performed on a subset of data. To load a fully trained PEFT model, place the PEFT checkpoint directory inside the current working directory before running the next cell.

Prepare this model by adding an adapter to the original FLAN-T5 model. You are setting `is_trainable=False` because the plan is only to perform inference with this PEFT model. If you were preparing the model for further training, you would set `is_trainable=True`.

In [ ]:
from peft import PeftModel, PeftConfig

peft_model_base = load_seq2seq_model(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

peft_model_path = pick_path(PEFT_CHECKPOINT_CANDIDATES + ["./peft-dialogue-summary-checkpoint-local"])

peft_model = PeftModel.from_pretrained(
    peft_model_base,
    peft_model_path,
    is_trainable=False,
)

peft_model = peft_model.to(device)

print(f"Using PEFT checkpoint: {peft_model_path}")


The number of trainable parameters will be `0` due to `is_trainable=False` setting:

In [ ]:
print(print_number_of_trainable_model_parameters(peft_model))

<a name='3.3'></a>
### 3.3 - Evaluate the Model Qualitatively (Human Evaluation) (2 points)

Make inferences for the same example as in sections [1.3](#1.3) and [2.3](#2.3), with the original model, fully fine-tuned and PEFT model.

In [ ]:
index = 200
dialogue = dataset["test"][index]["dialogue"]
human_baseline_summary = dataset["test"][index]["summary"]

prompt = build_prompt(dialogue)

original_model_text_output = generate_summary(original_model, prompt)
instruct_model_text_output = generate_summary(instruct_model, prompt)
peft_model_text_output = generate_summary(peft_model, prompt)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{human_baseline_summary}")
print(dash_line)
print(f"ORIGINAL MODEL:\n{original_model_text_output}")
print(dash_line)
print(f"INSTRUCT MODEL:\n{instruct_model_text_output}")
print(dash_line)
print(f"PEFT MODEL:\n{peft_model_text_output}")


<a name='3.4'></a>
### 3.4 - Evaluate the Model Quantitatively (with ROUGE Metric) (3 points)
Perform inferences for the sample of the test dataset (only 10 dialogues and summaries to save time).

In [ ]:
dialogues = dataset["test"][0:10]["dialogue"]
human_baseline_summaries = dataset["test"][0:10]["summary"]

original_model_summaries = []
instruct_model_summaries = []
peft_model_summaries = []

for dialogue in dialogues:
    prompt = build_prompt(dialogue)

    original_model_summaries.append(generate_summary(original_model, prompt))
    instruct_model_summaries.append(generate_summary(instruct_model, prompt))
    peft_model_summaries.append(generate_summary(peft_model, prompt))

zipped_summaries = list(
    zip(
        human_baseline_summaries,
        original_model_summaries,
        instruct_model_summaries,
        peft_model_summaries,
    )
)

df = pd.DataFrame(
    zipped_summaries,
    columns=[
        "human_baseline_summaries",
        "original_model_summaries",
        "instruct_model_summaries",
        "peft_model_summaries",
    ],
)
df


In [ ]:
rouge = evaluate.load("rouge")

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries,
)

print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)
print("PEFT MODEL:")
print(peft_model_results)


Notice, that PEFT model results are not too bad, while the training process was much easier!

You already computed ROUGE score on the full dataset using `dialogue-summary-training-results.csv` from the current working directory. The next cell reuses that dataframe to compare the original, full fine-tuned, and PEFT models on the larger saved result set.


In [ ]:
human_baseline_summaries = results["human_baseline_summaries"].values
original_model_summaries = results["original_model_summaries"].values
instruct_model_summaries = results["instruct_model_summaries"].values
peft_model_summaries = results["peft_model_summaries"].values

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries,
)

print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)
print("PEFT MODEL:")
print(peft_model_results)


The results show less of an improvement over full fine-tuning, but the benefits of PEFT typically outweigh the slightly-lower performance metrics.

Calculate the improvement of PEFT over the original model:

In [ ]:
print("Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(original_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Now calculate the improvement of PEFT over a full fine-tuned model:

In [ ]:
print("Absolute percentage improvement of PEFT MODEL over INSTRUCT MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(instruct_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Here you should see a small ROUGE decrease for PEFT versus the full fine-tuned model, while still retaining a clear improvement over the original zero-shot baseline. That trade-off is the main takeaway for the written discussion: PEFT gives up a little quality in exchange for a much cheaper training setup.
